In [2]:
pip install lightgbm

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_validate
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, roc_curve
from lightgbm import LGBMClassifier

import sklearn
import lightgbm as lgb
print('scikit-learn :', sklearn.__version__)
print('lightgbm     :', lgb.__version__)

TRAIN_PATH   = 'train.csv'
TEST_PATH    = 'secret_test_features'
RANDOM_STATE = 42

scikit-learn : 1.7.1
lightgbm     : 4.6.0


Load & Cek Data

In [4]:
df = pd.read_csv(TRAIN_PATH)

print(f'Shape  : {df.shape}')
print(f'Nulls  : {df.isnull().sum().sum()}')
print('\nTarget distribution:')
print(df['target'].value_counts())
print(f'\nImbalance ratio: {df["target"].value_counts()[0]/df["target"].value_counts()[1]:.2f}:1')
df.head(3)

Shape  : (38000, 18)
Nulls  : 0

Target distribution:
target
0    23000
1    15000
Name: count, dtype: int64

Imbalance ratio: 1.53:1


,id,num_1,num_2,num_3,num_4,num_5,num_6,num_7,num_8,num_9,cat_1,cat_2,cat_3,cat_4,cat_5,cat_6,cat_7,target
0,0,68,15136,148531,491,80,4,14.98,12,0.55,Bachelor's,Self-employed,Divorced,No,No,Auto,Yes,1
1,1,66,148756,40664,825,16,4,11.32,48,0.25,Bachelor's,Unemployed,Single,Yes,No,Other,No,0
2,2,61,42190,138134,378,5,1,20.50,48,0.79,Bachelor's,Unemployed,Married,Yes,No,Education,Yes,0


In [5]:
def add_features(df):
    """Domain-aware feature engineering untuk loan default prediction."""
    df = df.copy()
    # Financial ratios
    df['loan_to_income']      = df['num_3'] / (df['num_2'] + 1)           # pinjaman / pendapatan
    df['credit_util_ratio']   = df['num_3'] / (df['num_4'] + 1)           # pinjaman / skor kredit
    df['debt_burden']         = df['num_9'] * df['num_3']                 # DTI ratio * pinjaman
    df['income_per_month']    = df['num_2'] / (df['num_5'] + 1)           # pendapatan / bulan bekerja
    # Loan cost
    df['interest_x_months']   = df['num_7'] * df['num_8']                 # interest * tenor
    df['total_cost']          = df['num_3'] * (1 + df['num_7'] / 100 * df['num_8'] / 12)  # estimasi total bayar
    df['monthly_payment_est'] = df['total_cost'] / (df['num_8'] + 1)      # estimasi cicilan bulanan
    df['payment_to_income']   = df['monthly_payment_est'] / (df['num_2'] / 12 + 1)        # beban cicilan
    # Credit profile
    df['age_x_credit']        = df['num_1'] * df['num_4']                 # umur * skor kredit
    return df

df = add_features(df)
print(f'Total features setelah FE: {df.shape[1] - 2}')  # excl. id & target

Total features setelah FE: 25


Split Train / Validation

In [6]:
CAT_COLS = [f'cat_{i}' for i in range(1, 8)]
NUM_COLS = [c for c in df.columns if c not in CAT_COLS + ['id', 'target']]

X = df.drop(columns=['id', 'target'])
y = df['target']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f'Train : {X_train.shape} | Pos={y_train.sum()} Neg={(y_train==0).sum()}')
print(f'Val   : {X_val.shape}   | Pos={y_val.sum()} Neg={(y_val==0).sum()}')
print(f'Num features : {len(NUM_COLS)}')
print(f'Cat features : {len(CAT_COLS)}')

Train : (30400, 25) | Pos=12000 Neg=18400
Val   : (7600, 25)   | Pos=3000 Neg=4600
Num features : 18
Cat features : 7


Preprocessor

In [7]:
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), NUM_COLS),
    ('cat', OrdinalEncoder(
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ), CAT_COLS)
], remainder='drop')

print('Preprocessor ready.')
print(f'  Numerical  : {len(NUM_COLS)} cols')
print(f'  Categorical: {len(CAT_COLS)} cols')

Preprocessor ready.
  Numerical  : 18 cols
  Categorical: 7 cols


Model 1 — HistGradientBoostingClassifier


In [8]:
pipe_hgb = Pipeline([
    ('pre', preprocessor),
    ('clf', HistGradientBoostingClassifier(
        class_weight='balanced',
        random_state=RANDOM_STATE
    ))
])

param_hgb = {
    'clf__max_iter'          : [200, 300, 500],
    'clf__learning_rate'     : [0.03, 0.05, 0.1, 0.2],
    'clf__max_depth'         : [3, 4, 5, 6],
    'clf__min_samples_leaf'  : [20, 40, 60, 100],
    'clf__l2_regularization' : [0.0, 0.1, 0.5, 1.0],
    'clf__max_leaf_nodes'    : [15, 31, 63, None],
}

search_hgb = RandomizedSearchCV(
    pipe_hgb, param_hgb,
    n_iter=40, cv=5, scoring='roc_auc',
    n_jobs=-1, verbose=1, random_state=RANDOM_STATE
)
search_hgb.fit(X_train, y_train)
best_hgb = search_hgb.best_estimator_

auc_hgb = roc_auc_score(y_val, best_hgb.predict_proba(X_val)[:, 1])
f1_hgb  = f1_score(y_val, best_hgb.predict(X_val))
acc_hgb = accuracy_score(y_val, best_hgb.predict(X_val))

print('=== HistGradientBoosting ===')
print(f'Best CV AUC  : {search_hgb.best_score_:.4f}')
print(f'Best params  : {search_hgb.best_params_}')
print(f'Val AUC-ROC  : {auc_hgb:.4f}')
print(f'Val F1-Score : {f1_hgb:.4f}')
print(f'Val Accuracy : {acc_hgb:.4f}')

Fitting 5 folds for each of 40 candidates, totalling 200 fits
=== HistGradientBoosting ===
Best CV AUC  : 0.7529
Best params  : {'clf__min_samples_leaf': 20, 'clf__max_leaf_nodes': 63, 'clf__max_iter': 200, 'clf__max_depth': 3, 'clf__learning_rate': 0.1, 'clf__l2_regularization': 1.0}
Val AUC-ROC  : 0.7519
Val F1-Score : 0.6406
Val Accuracy : 0.6911


Cross Validation — HistGradientBoosting

In [9]:
cv_hgb = cross_validate(
    best_hgb, X, y, cv=5,
    scoring=['roc_auc', 'f1', 'accuracy'],
    n_jobs=-1
)

print('=== CV HistGradientBoosting (5-fold) ===')
print(f'AUC-ROC  : {cv_hgb["test_roc_auc"].mean():.4f} ± {cv_hgb["test_roc_auc"].std():.4f}')
print(f'F1-Score : {cv_hgb["test_f1"].mean():.4f} ± {cv_hgb["test_f1"].std():.4f}')
print(f'Accuracy : {cv_hgb["test_accuracy"].mean():.4f} ± {cv_hgb["test_accuracy"].std():.4f}')

=== CV HistGradientBoosting (5-fold) ===
AUC-ROC  : 0.7535 ± 0.0053
F1-Score : 0.6332 ± 0.0049
Accuracy : 0.6886 ± 0.0042


Model 2 — LightGBM


In [10]:
pipe_lgbm = Pipeline([
    ('pre', preprocessor),
    ('clf', LGBMClassifier(
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1
    ))
])

param_lgbm = {
    'clf__n_estimators'     : [200, 300, 500, 700],
    'clf__learning_rate'    : [0.03, 0.05, 0.1, 0.15],
    'clf__max_depth'        : [3, 4, 5, 6, -1],
    'clf__num_leaves'       : [15, 31, 63, 127],
    'clf__min_child_samples': [20, 50, 100],
    'clf__subsample'        : [0.6, 0.7, 0.8, 1.0],
    'clf__colsample_bytree' : [0.6, 0.7, 0.8, 1.0],
    'clf__reg_alpha'        : [0.0, 0.01, 0.1],
    'clf__reg_lambda'       : [0.0, 0.1, 1.0],
}

search_lgbm = RandomizedSearchCV(
    pipe_lgbm, param_lgbm,
    n_iter=40, cv=5, scoring='roc_auc',
    n_jobs=-1, verbose=1, random_state=RANDOM_STATE
)
search_lgbm.fit(X_train, y_train)
best_lgbm = search_lgbm.best_estimator_

auc_lgbm = roc_auc_score(y_val, best_lgbm.predict_proba(X_val)[:, 1])
f1_lgbm  = f1_score(y_val, best_lgbm.predict(X_val))
acc_lgbm = accuracy_score(y_val, best_lgbm.predict(X_val))

print('=== LightGBM ===')
print(f'Best CV AUC  : {search_lgbm.best_score_:.4f}')
print(f'Best params  : {search_lgbm.best_params_}')
print(f'Val AUC-ROC  : {auc_lgbm:.4f}')
print(f'Val F1-Score : {f1_lgbm:.4f}')
print(f'Val Accuracy : {acc_lgbm:.4f}')

Fitting 5 folds for each of 40 candidates, totalling 200 fits
=== LightGBM ===
Best CV AUC  : 0.7527
Best params  : {'clf__subsample': 0.7, 'clf__reg_lambda': 0.0, 'clf__reg_alpha': 0.01, 'clf__num_leaves': 31, 'clf__n_estimators': 300, 'clf__min_child_samples': 50, 'clf__max_depth': 3, 'clf__learning_rate': 0.03, 'clf__colsample_bytree': 0.6}
Val AUC-ROC  : 0.7516
Val F1-Score : 0.6353
Val Accuracy : 0.6859


Cross Validation — LightGBM

In [11]:
cv_lgbm = cross_validate(
    best_lgbm, X, y, cv=5,
    scoring=['roc_auc', 'f1', 'accuracy'],
    n_jobs=-1
)

print('=== CV LightGBM (5-fold) ===')
print(f'AUC-ROC  : {cv_lgbm["test_roc_auc"].mean():.4f} ± {cv_lgbm["test_roc_auc"].std():.4f}')
print(f'F1-Score : {cv_lgbm["test_f1"].mean():.4f} ± {cv_lgbm["test_f1"].std():.4f}')
print(f'Accuracy : {cv_lgbm["test_accuracy"].mean():.4f} ± {cv_lgbm["test_accuracy"].std():.4f}')

=== CV LightGBM (5-fold) ===
AUC-ROC  : 0.7541 ± 0.0050
F1-Score : 0.6335 ± 0.0053
Accuracy : 0.6873 ± 0.0049


Model 3 — Random Forest


In [12]:
pipe_rf = Pipeline([
    ('pre', preprocessor),
    ('clf', RandomForestClassifier(
        class_weight='balanced',
        n_jobs=-1,
        random_state=RANDOM_STATE
    ))
])

param_rf = {
    'clf__n_estimators'    : [200, 300, 500],
    'clf__max_depth'       : [5, 10, 15, 20, None],
    'clf__min_samples_leaf': [5, 10, 20, 50],
    'clf__max_features'    : ['sqrt', 'log2', 0.3, 0.5],
    'clf__max_samples'     : [0.7, 0.8, 0.9, None],
}

search_rf = RandomizedSearchCV(
    pipe_rf, param_rf,
    n_iter=40, cv=5, scoring='roc_auc',
    n_jobs=-1, verbose=1, random_state=RANDOM_STATE
)
search_rf.fit(X_train, y_train)
best_rf = search_rf.best_estimator_

auc_rf = roc_auc_score(y_val, best_rf.predict_proba(X_val)[:, 1])
f1_rf  = f1_score(y_val, best_rf.predict(X_val))
acc_rf = accuracy_score(y_val, best_rf.predict(X_val))

print('=== Random Forest ===')
print(f'Best CV AUC  : {search_rf.best_score_:.4f}')
print(f'Best params  : {search_rf.best_params_}')
print(f'Val AUC-ROC  : {auc_rf:.4f}')
print(f'Val F1-Score : {f1_rf:.4f}')
print(f'Val Accuracy : {acc_rf:.4f}')

Fitting 5 folds for each of 40 candidates, totalling 200 fits


KeyboardInterrupt: 

Cross Validation — Random Forest

In [ ]:
cv_rf = cross_validate(
    best_rf, X, y, cv=5,
    scoring=['roc_auc', 'f1', 'accuracy'],
    n_jobs=-1
)

print('=== CV Random Forest (5-fold) ===')
print(f'AUC-ROC  : {cv_rf["test_roc_auc"].mean():.4f} ± {cv_rf["test_roc_auc"].std():.4f}')
print(f'F1-Score : {cv_rf["test_f1"].mean():.4f} ± {cv_rf["test_f1"].std():.4f}')
print(f'Accuracy : {cv_rf["test_accuracy"].mean():.4f} ± {cv_rf["test_accuracy"].std():.4f}')

Ringkasan Hasil & Pilih Best Model


In [ ]:
results = pd.DataFrame([
    {
        'Model'      : 'HistGradientBoosting',
        'CV AUC'     : cv_hgb['test_roc_auc'].mean(),
        'CV AUC std' : cv_hgb['test_roc_auc'].std(),
        'Val AUC'    : auc_hgb,
        'Val F1'     : f1_hgb,
        'Val Acc'    : acc_hgb,
    },
    {
        'Model'      : 'LightGBM',
        'CV AUC'     : cv_lgbm['test_roc_auc'].mean(),
        'CV AUC std' : cv_lgbm['test_roc_auc'].std(),
        'Val AUC'    : auc_lgbm,
        'Val F1'     : f1_lgbm,
        'Val Acc'    : acc_lgbm,
    },
    {
        'Model'      : 'RandomForest',
        'CV AUC'     : cv_rf['test_roc_auc'].mean(),
        'CV AUC std' : cv_rf['test_roc_auc'].std(),
        'Val AUC'    : auc_rf,
        'Val F1'     : f1_rf,
        'Val Acc'    : acc_rf,
    },
]).sort_values('CV AUC', ascending=False).reset_index(drop=True)

print(results.to_string(index=False, float_format='{:.4f}'.format))

best_model_name = results.iloc[0]['Model']
print(f'\nBest Model (by CV AUC): {best_model_name}')

ROC Curve Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

models_info = [
    (best_hgb,  'HistGradientBoosting', '#E53935'),
    (best_lgbm, 'LightGBM',             '#FB8C00'),
    (best_rf,   'RandomForest',          '#43A047'),
]

for model, name, color in models_info:
    y_prob = model.predict_proba(X_val)[:, 1]
    fpr, tpr, _ = roc_curve(y_val, y_prob)
    auc = roc_auc_score(y_val, y_prob)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1.2, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve Comparison — Loan Default Prediction', fontsize=13)
ax.legend(loc='lower right', fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_comparison.png', dpi=150)
plt.show()
print('Plot saved: roc_comparison.png')

In [ ]:
import os

trained_models = {
    'HistGradientBoosting': best_hgb,
    'LightGBM'            : best_lgbm,
    'RandomForest'        : best_rf,
}
chosen_model = trained_models[best_model_name]

if os.path.exists(TEST_PATH):
    df_test = pd.read_csv(TEST_PATH)
    df_test = add_features(df_test)                      # same FE as training
    X_test  = df_test.drop(columns=['id'], errors='ignore')

    prob = chosen_model.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.5).astype(int)

    submission = pd.DataFrame({
        'id'         : df_test['id'],
        'prediction' : pred,
        'probability': prob.round(6),
    })
    submission.to_csv('nama_team_predictions.csv', index=False)

    print(f'Submission saved: nama_team_predictions.csv')
    print(f'   Model      : {best_model_name}')
    print(f'   Rows       : {len(submission)}')
    print(f'   Predicted 1: {pred.sum()} ({pred.mean()*100:.1f}%)')
    print()
    print(submission.head())
else:
    print(f'File "{TEST_PATH}" not found.')
    print('   Letakkan test.csv di folder yang sama, lalu jalankan ulang cell ini.')